# 1.2 — Разведочный анализ данных

**Папка 1, подноутбук 2.** Разведочный анализ полного геотехнического набора:
гранулометрические кривые и классификации (ГОСТ, PLAXIS, тип отклика), распределения
физических и механических параметров по датасету и **внутри типов грунта**, корреляции,
PCA и связь нагрузки с откликом. Рисунки и таблицы — на английском.

## Окружение и загрузка артефакта

In [2]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from liquefaction_ai.viz import (bar, box_grid, correlation_heatmap, grouped_bar,
                                 histogram_grid, lines, scatter, scatter_by_group, surface3d_grid)
from liquefaction_ai import load_population_artifact
from liquefaction_ai.constants import (FEATURE_AXIS_LABELS_EN, LOAD_DISPLAY_NAMES_EN,
                                       LOAD_NAMES, SOIL_DISPLAY_NAMES_EN, SOIL_NAMES)
from liquefaction_ai.data.grainsize import FRACTION_BOUNDS, FRACTION_KEYS, TYPE_GROUND_NAMES_EN

population, config = load_population_artifact(DATA_DIR)
meta = population["meta"].copy()
meta["log10_permeability"] = np.log10(meta["permeability"])
meta["soil_en"] = meta["soil_type"].map(SOIL_DISPLAY_NAMES_EN)
meta["load_en"] = meta["load_mode"].map(LOAD_DISPLAY_NAMES_EN)
# Совместимость с реальными данными: синтетические поля заменяем наблюдаемым прокси (пик PPR)
for _col in ["damage_max_true"]:
    if _col not in meta.columns:
        meta[_col] = meta["PPR_max_true"]
soil_order = list(SOIL_NAMES); load_order = list(LOAD_NAMES)
soil_present = [s for s in soil_order if (meta["soil_type"] == s).any()]
print("Loaded scenarios:", len(meta))

Loaded scenarios: 666


## Классификации: тип грунта (ГОСТ), класс PLAXIS, тип отклика

In [3]:
tg = meta["type_ground"].map(TYPE_GROUND_NAMES_EN).value_counts()
bar(list(tg.index), tg.values, title="Soil-type composition (GOST type_ground)",
    ylabel="N scenarios", text_fmt=".0f", color="#0b6efd", horizontal=True,
    save=SAVE_FIGS, fig_id="1_2_type_ground").show()

pc = meta["plaxis_class"].value_counts()
bar(list(pc.index), pc.values, title="PLAXIS grain-size class distribution",
    ylabel="N scenarios", text_fmt=".0f", color="#198754",
    save=SAVE_FIGS, fig_id="1_2_plaxis_class").show()

rt = meta["response_type"].value_counts()
bar(list(rt.index), rt.values, title="Cyclic response-type distribution",
    ylabel="N scenarios", text_fmt=".0f", color="#d63384",
    save=SAVE_FIGS, fig_id="1_2_response_type").show()

[save_figure] PNG для '1_2_type_ground' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[save_figure] PNG для '1_2_plaxis_class' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[save_figure] PNG для '1_2_response_type' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Гранулометрические кривые по типам грунта

Средняя кривая «процент прохода» для каждого типа грунта (логарифмическая ось диаметра).
Кривые расходятся от крупнозернистых песков к тонкодисперсным глинам.

In [4]:
gran_cols = [f"gran_{k}" for k in FRACTION_KEYS]
diameters = np.array([FRACTION_BOUNDS[k] for k in FRACTION_KEYS])      # убывает 20..0.002
fr = meta[gran_cols].to_numpy()
passing = np.cumsum(fr[:, ::-1], axis=1)[:, ::-1]                      # % мельче diameters[i]
d_asc = diameters[::-1]
series = []
for s in soil_present:
    m = meta["soil_type"].to_numpy() == s
    series.append({"x": d_asc, "y": passing[m].mean(axis=0)[::-1], "name": SOIL_DISPLAY_NAMES_EN[s]})
lines(series, title="Mean grain-size distribution curves by soil type",
      xlabel="Particle diameter (mm)", ylabel="Percent passing (%)", logx=True,
      save=SAVE_FIGS, fig_id="1_2_grain_size_curves").show()

[save_figure] PNG для '1_2_grain_size_curves' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Распределения параметров по всему датасету

In [5]:
cols = ["e", "D_r", "I_p", "V_s", "fines_content", "clay_fraction", "Cu", "log10_permeability"]
histogram_grid(meta, columns=cols, titles=[FEATURE_AXIS_LABELS_EN.get(c, c) for c in cols],
               n_cols=3, title="Parameter distributions across the whole dataset",
               save=SAVE_FIGS, fig_id="1_2_param_distributions").show()

[save_figure] PNG для '1_2_param_distributions' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Распределения физических параметров внутри типов грунта

In [6]:
cols = ["e", "D_r", "fines_content", "clay_fraction", "Cu", "log10_permeability"]
box_grid(meta, value_cols=cols, group_col="soil_type", group_order=soil_present,
         group_labels=SOIL_DISPLAY_NAMES_EN, titles=[FEATURE_AXIS_LABELS_EN.get(c, c) for c in cols],
         n_cols=3, title="Physical-parameter distributions within soil types",
         save=SAVE_FIGS, fig_id="1_2_params_by_soil").show()

[save_figure] PNG для '1_2_params_by_soil' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Механический отклик внутри типов грунта

In [7]:
cols = ["PPR_max_true", "N_liq_true", "phi", "cohesion", "E_modulus"]
box_grid(meta, value_cols=cols, group_col="soil_type", group_order=soil_present,
         group_labels=SOIL_DISPLAY_NAMES_EN, titles=[FEATURE_AXIS_LABELS_EN.get(c, c) for c in cols],
         n_cols=3, title="Mechanical response and strength within soil types",
         save=SAVE_FIGS, fig_id="1_2_response_by_soil").show()

[save_figure] PNG для '1_2_response_by_soil' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Корреляционная структура

In [8]:
corr_cols = ["e", "D_r", "I_p", "Il", "V_s", "fines_content", "clay_fraction", "sigma_eff",
             "K0", "crr_ref", "PPR_max_true", "N_liq_true", "liq_label"]
corr_cols = [c for c in corr_cols if c in meta.columns]
correlation_heatmap(meta[corr_cols].corr(), title="Pearson correlations of parameters and targets",
                    save=SAVE_FIGS, fig_id="1_2_correlation").show()

[save_figure] PNG для '1_2_correlation' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Проекция PCA

In [9]:
pca_cols = ["e", "D_r", "I_p", "V_s", "fines_content", "clay_fraction", "sigma_eff", "Vs1"]
pca_cols = [c for c in pca_cols if c in meta.columns]
X = StandardScaler().fit_transform(meta[pca_cols].to_numpy())
pca = PCA(n_components=2, random_state=config.seed); proj = pca.fit_transform(X)
evr = pca.explained_variance_ratio_
meta["PC1"], meta["PC2"] = proj[:, 0], proj[:, 1]
scatter_by_group(meta, "PC1", "PC2", group_col="soil_en",
                 group_order=[SOIL_DISPLAY_NAMES_EN[s] for s in soil_present],
                 title=f"PCA by soil type (PC1 {evr[0]*100:.1f}%, PC2 {evr[1]*100:.1f}%)",
                 xlabel=f"PC1 ({evr[0]*100:.1f}%)", ylabel=f"PC2 ({evr[1]*100:.1f}%)",
                 save=SAVE_FIGS, fig_id="1_2_pca_by_soil").show()
scatter(proj[:, 0], proj[:, 1], color=meta["PPR_max_true"], color_label="Peak PPR (–)",
        title="PCA coloured by liquefaction risk", xlabel=f"PC1 ({evr[0]*100:.1f}%)",
        ylabel=f"PC2 ({evr[1]*100:.1f}%)", save=SAVE_FIGS, fig_id="1_2_pca_by_risk").show()

[save_figure] PNG для '1_2_pca_by_soil' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[save_figure] PNG для '1_2_pca_by_risk' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Связь нагрузки и отклика, поверхности риска

In [10]:
scatter(meta["CSR_max"], meta["PPR_max_true"], color=meta["PPR_max_true"],
        color_label="Liquefaction risk (–)", title="Peak CSR vs peak PPR",
        xlabel="Peak CSR (–)", ylabel="Peak pore-pressure ratio (–)", hline=1.0,
        save=SAVE_FIGS, fig_id="1_2_csr_vs_ppr").show()

sample = meta.sample(min(3000, len(meta)), random_state=config.seed)
specs = [
    {"x": sample["D_r"], "y": sample["CSR_base"], "z": sample["PPR_max_true"],
     "title": "Dr, CSR → risk", "xlabel": "Dr", "ylabel": "CSR", "zlabel": "Risk", "colorscale": "Viridis"},
    {"x": sample["fines_content"], "y": sample["CSR_base"], "z": sample["PPR_max_true"],
     "title": "FC, CSR → peak PPR", "xlabel": "Fines %", "ylabel": "CSR", "zlabel": "Peak PPR", "colorscale": "Plasma"},
    {"x": sample["Vs1"], "y": sample["CSR_base"], "z": np.log10(sample["N_liq_true"] + 1.0),
     "title": "Vs1, CSR → log10 N_liq", "xlabel": "Vs1 (m/s)", "ylabel": "CSR", "zlabel": "log10 N_liq", "colorscale": "Cividis"},
]
surface3d_grid(specs, n_cols=3, title="Three-dimensional liquefaction-risk surfaces",
               save=SAVE_FIGS, fig_id="1_2_risk_surfaces").show()

[save_figure] PNG для '1_2_csr_vs_ppr' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[save_figure] PNG для '1_2_risk_surfaces' не сохранён (нет движка экспорта): 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



## Итог

Проанализирован полный геотехнический набор. Следующий шаг — **1.3 анализ параметров CRR**.